In [ ]:
import threading
import sys
import time
import pyaudio
import numpy as np
import matplotlib.pyplot as plt
import wave
import speech_recognition as sr
from speech_recognition import AudioData

# Event to signal when recording should stop
stop_event = threading.Event()

# Function to wait for user input to stop recording
def wait_for_enter():
    input("\n🎤 Press Enter to stop recording...\n")
    stop_event.set() # Set the event to stop the recording loop

# Function to display a spinning indicator during recording
def spinner():
    chars = '|/-\\'
    i = 0
    while not stop_event.is_set(): # Keep spinning until stop_event is set
        sys.stdout.write(f'\r🔴 Recording... {chars[i % 4]}') # Update spinner character
        sys.stdout.flush()
        i += 1
        time.sleep(0.1)
    print("\r✅ Recording complete!          ") # Clear spinner and print completion message

# Function to record audio from the microphone
def record_audio():
    p = pyaudio.PyAudio()
    # Open audio stream for recording
    stream = p.open(format=pyaudio.paInt16, channels=1, rate=16000,
                    input=True, frames_per_buffer=1024)
    frames = []

    # Start threads for stopping recording and showing spinner
    threading.Thread(target=wait_for_enter, daemon=True).start()
    threading.Thread(target=spinner, daemon=True).start()

    # Read audio data in chunks until stop_event is set
    while not stop_event.is_set():
        frames.append(stream.read(1024))

    # Stop and close the audio stream
    stream.stop_stream()
    stream.close()
    width = p.get_sample_size(pyaudio.paInt16)
    p.terminate()
    return b''.join(frames), 16000, width # Return audio data, sample rate, and sample width

# Function to save recorded audio to a WAV file
def save_audio(data, rate, width, filename="recording.wav"):
    with wave.open(filename, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(width)
        wf.setframerate(rate)
        wf.writeframes(data)
    print(f"💾 Saved: {filename}")

# Function to transcribe audio data using Google Web Speech API
def transcribe(data, rate, width):
    recognizer = sr.Recognizer()
    # Convert raw audio data to AudioData object
    audio = AudioData(data, rate, width)
    try:
        text = recognizer.recognize_google(audio) # Perform speech recognition
        print(f"📝 Transcription: {text}")
    except sr.UnknownValueError:
        print("❌ Could not understand audio") # Handle unintelligible audio
    except sr.RequestError as e:
        print(f"❌ API Error: {e}") # Handle API errors

# Function to plot the waveform of the recorded audio
def plot_waveform(data, rate):
    samples = np.frombuffer(data, dtype=np.int16) # Convert bytes to numpy array of samples
    time_axis = np.linspace(0, len(samples) / rate, len(samples)) # Create time axis
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, samples, color='blue')
    plt.title("Your Voice Waveform")
    plt.xlabel("Time (seconds)")
    plt.ylabel("Amplitude")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Main function to orchestrate the recording, saving, transcribing, and plotting process
def main():
    print("=" * 40)
    print("🎙️  HELLO AI, CAN YOU HEAR ME?")
    print("=" * 40)
    print("\nSpeak into your microphone...")

    # Reset the stop_event for a new recording session
    stop_event.clear()

    # Record audio
    audio_data, rate, width = record_audio()
    # Save audio to file
    save_audio(audio_data, rate, width)
    # Transcribe the audio
    transcribe(audio_data, rate, width)
    # Plot the audio waveform
    plot_waveform(audio_data, rate)

# Entry point of the script
if __name__ == "__main__":
    main()